In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- Helper Functions ---
def sigmoid(z_):
        return 1/(1+np.exp(-z_))
def softmax(z):
    z = z - np.max(z, axis=0, keepdims=True)
    return np.exp(z) / (np.sum(np.exp(z), axis=0, keepdims=True) + 1e-10)

def relu(z):
    return np.maximum(z, 0)

def r_derivative(z):
    return np.where(z > 0, 1, 0)

# --- Data Loading & Splitting ---
df = pd.read_csv("mnist_train.csv")
df1 = pd.read_csv("mnist_test.csv")

# Train Data
X_train = df.iloc[:, 1:].values
y_train_idx = df["label"].values
y_train_onehot = np.eye(10)[y_train_idx].T

# Test Data
X_test = df1.iloc[:, 1:].values
y_test_idx = df1["label"].values
y_test_onehot = np.eye(10)[y_test_idx].T

# --- Scaling & PCA ---
X_train_scaled = X_train / 255.0
X_test_scaled = X_test / 255.0

pca = PCA(n_components=0.95)
X = pca.fit_transform(X_train_scaled)
X_test = pca.transform(X_test_scaled)

# --- Neural Network Setup ---
layers = [512, 256, 128, 10]
activations = ["relu", "relu", "softmax"]
learning_rate = 0.01
m = X.shape[0]

W = []
b = []
prev_units = X.shape[1]

for L in range(len(layers)):
    w = np.random.randn(layers[L], prev_units) * np.sqrt(2 / prev_units)
    b_ = np.zeros((layers[L], 1))
    W.append(w)
    b.append(b_)
    prev_units = layers[L]

cost_history, cost_history_test = [], []
acc_history, acc_history_test = [], []

# --- Training Loop ---
for epoch in range(1000):
    # Forward Prop
    A_ = X.T
    A = [A_]
    Z = []
    
    for i in range(len(layers)):
        z = W[i] @ A[-1] + b[i]
        Z.append(z)
        if i == len(layers) - 1:
            A_ = softmax(z)
        else:
            A_ = relu(z) if activations[i] == "relu" else sigmoid(z)
        A.append(A_)

    # Cost
    cost = -np.sum(y_train_onehot * np.log(A[-1] + 1e-10)) / m
    cost_history.append(cost)
    acc = np.mean(np.argmax(A[-1], axis=0) == y_train_idx) * 100
    acc_history.append(acc)

    # Backprop
    delta = A[-1] - y_train_onehot
    for i in range(len(layers) - 1, -1, -1):
        dw = (delta @ A[i].T) / m
        db = np.sum(delta, axis=1, keepdims=True) / m
        dw = np.clip(dw, -1, 1) 
        db = np.clip(db, -1, 1)
        if i > 0:
            delta = (W[i].T @ delta) * r_derivative(Z[i-1])
        
        W[i] -= learning_rate * dw
        b[i] -= learning_rate * db

    # Test Eval
    A_test = X_test.T
    for i in range(len(layers)):
        z = W[i] @ A_test + b[i]
        A_test = softmax(z) if i == len(layers) - 1 else (relu(z) if activations[i] == "relu" else sigmoid(z))
    
    test_cost = -np.sum(y_test_onehot * np.log(A_test + 1e-10)) / X_test.shape[0]
    test_acc = np.mean(np.argmax(A_test, axis=0) == y_test_idx) * 100
    
    cost_history_test.append(test_cost)
    acc_history_test.append(test_acc)
        
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Train Acc: {acc:.2f}% | Test Acc: {test_acc:.2f}%")

plt.figure(figsize=(10,4))
plt.plot(acc_history, label="Train Acc"); plt.plot(acc_history_test, label="Test Acc"); plt.legend(); plt.show()